# BaseML 机器学习测试
## 传统机器学习算法实战

本 Notebook 演示如何使用 XEdu BaseML 模块进行传统机器学习任务。

### 学习目标
- ✅ 了解常见机器学习算法原理
- ✅ 掌握 BaseML API 使用方法
- ✅ 完成数据预处理和特征工程
- ✅ 训练并评估多个机器学习模型

### 预计用时：20-25 分钟

In [ ]:
# 导入必要的库
import baseml
from baseml import RandomForest, SVM, KNN
from basedt import Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"BaseML 版本：{baseml.__version__}")
print("✅ 库导入成功")

## 步骤 1: 加载和探索数据集

使用经典的鸢尾花 (Iris) 数据集进行分类任务

In [ ]:
# 加载 Iris 数据集
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

print(f"数据集信息:")
print(f"  - 样本数：{len(X)}")
print(f"  - 特征数：{X.shape[1]}")
print(f"  - 类别数：{len(np.unique(y))}")
print(f"\n特征名称：{iris.feature_names}")
print(f"类别名称：{iris.target_names}")

# 查看前 5 个样本
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
print(f"\n前 5 个样本:")
print(df.head())

## 步骤 2: 数据预处理

划分训练集和测试集，并进行标准化处理

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集样本数：{len(X_train)}")
print(f"测试集样本数：{len(X_test)}")

# 数据标准化
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ 数据预处理完成")

## 步骤 3: 训练多个模型

分别训练随机森林、SVM 和 KNN 三个模型

In [ ]:
# 1. 随机森林分类器
print("训练随机森林模型...")
rf_model = RandomForest(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
print("✅ 随机森林训练完成")

# 2. 支持向量机 (SVM)
print("\n训练 SVM 模型...")
svm_model = SVM(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)
print("✅ SVM 训练完成")

# 3. K 近邻 (KNN)
print("\n训练 KNN 模型...")
knn_model = KNN(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
print("✅ KNN 训练完成")

## 步骤 4: 模型评估与对比

In [ ]:
# 在测试集上评估所有模型
models = {
    'Random Forest': rf_model,
    'SVM': svm_model,
    'KNN': knn_model
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    accuracy = (y_pred == y_test).mean()
    results.append({'Model': name, 'Accuracy': accuracy})
    
    print(f"\n{name}:")
    print(f"  准确率：{accuracy:.4f}")
    print(f"  分类报告:")
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 创建结果 DataFrame
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("模型性能对比:")
print(results_df.sort_values('Accuracy', ascending=False))

## 步骤 5: 可视化结果

In [ ]:
import matplotlib.pyplot as plt

# 绘制模型性能对比图
plt.figure(figsize=(10, 6))
plt.bar(results_df['Model'], results_df['Accuracy'], color=['#3498db', '#e74c3c', '#2ecc71'])
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.title('Model Performance Comparison')
plt.ylim(0.9, 1.0)
plt.grid(axis='y', alpha=0.3)

# 在柱子上标注数值
for i, v in enumerate(results_df['Accuracy']):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.savefig('ml_model_comparison.png', dpi=300)
plt.show()

# 绘制最佳模型的混淆矩阵
best_model_name = results_df.loc[results_df['Accuracy'].idxmax()]['Model']
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_best)
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.colorbar()

# 添加文本标签
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(range(3), iris.target_names)
plt.yticks(range(3), iris.target_names)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

print(f"📊 可视化结果已保存")

## 步骤 6: 特征重要性分析

In [ ]:
# 分析随机森林的特征重要性
if hasattr(rf_model, 'feature_importances_'):
    importances = rf_model.feature_importances_
    
    plt.figure(figsize=(10, 6))
    indices = np.argsort(importances)[::-1]
    plt.bar(range(len(importances)), importances[indices])
    plt.xticks(range(len(importances)), [iris.feature_names[i] for i in indices], rotation=45)
    plt.xlabel('Feature')
    plt.ylabel('Importance')
    plt.title('Feature Importances (Random Forest)')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300)
    plt.show()
    
    print("特征重要性排序:")
    for i in indices:
        print(f"  {iris.feature_names[i]}: {importances[i]:.4f}")

## 总结与拓展

### 本节所学
1. ✅ BaseML 机器学习 API 的基本用法
2. ✅ 数据预处理和特征工程技巧
3. ✅ 多模型训练和对比评估
4. ✅ 特征重要性分析方法

### 课后练习
1. 📝 尝试其他机器学习算法（如决策树、朴素贝叶斯）
2. 📝 使用交叉验证评估模型稳定性
3. 📝 调整超参数优化模型性能
4. 📝 使用真实业务数据进行分类

### 积分奖励
- ✅ 完成基础练习：+150 XP
- ✅ 准确率达到 95% 以上：+250 XP
- ✅ 完成特征分析：+100 XP

### 下一步
继续学习下一个 Notebook: `04_xeduhub_model_zoo.ipynb`